In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Skin', 'Inflamed', 'Delayed', 'Fail1', 'Inter1', 'Inter2', 'Fail2','IPS']

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome'

multiome = pd.read_csv(f'{indir}/csv/clusters/rna_leiden_v7.csv', sep='\t', header=0, index_col=0)
multiome.shape

In [ ]:
multiome = multiome[['time_num', 'coarse_v7']]
multiome['assay'] = 'multiome'
multiome.rename(columns={'coarse_v7' : 'cluster'}, inplace=True)

In [ ]:
snmct = pd.read_csv(f'{projdir}/csv/label_transfer/snmct_xgboost_v7.csv', sep='\t', header=0, index_col=0)
snmct.shape

In [ ]:
snmct = snmct[['time_num', 'pred_coarse']]
snmct.rename(columns={'pred_coarse' : 'cluster'}, inplace=True)

In [ ]:
snmct

In [ ]:
multiome_dist = pd.DataFrame(multiome['cluster'].value_counts().sort_index())
snmct_dist = pd.DataFrame(snmct['cluster'].value_counts().sort_index())

In [ ]:
dists = pd.concat([multiome_dist, snmct_dist], axis=1)
dists.columns = ['multiome', 'snmCT']

In [ ]:
dists.head()

In [ ]:
props = dists.div(dists.sum(axis=0))
props.head()

In [ ]:
times = sorted(multiome['time_num'].unique())
clusters = sorted(multiome['cluster'].unique())
multiome_dist = pd.DataFrame(index=times, columns=clusters)

for time in times:
    tmp_multiome = multiome[multiome['time_num'] == time]
    multiome_dist.loc[time] = tmp_multiome['cluster'].value_counts().sort_index()
    
multiome_dist.fillna(0, inplace=True)
multiome_dist = multiome_dist.astype(int)
multiome_prop = multiome_dist.div(multiome_dist.sum(axis=1), axis=0)

In [ ]:
multiome_dist

In [ ]:
multiome_prop.head()

In [ ]:
times = sorted(snmct['time_num'].unique())
clusters = sorted(snmct['cluster'].unique())
snmct_dist = pd.DataFrame(index=times, columns=clusters)

for time in times:
    tmp_snmct = snmct[snmct['time_num'] == time]
    snmct_dist.loc[time] = tmp_snmct['cluster'].value_counts().sort_index()
    
snmct_dist.fillna(0, inplace=True)
snmct_dist = snmct_dist.astype(int)
snmct_prop = snmct_dist.div(snmct_dist.sum(axis=1), axis=0)

In [ ]:
snmct_dist

In [ ]:
snmct_prop

In [ ]:
snmct_prop.sum(axis=1)

In [ ]:
snmct_prop = snmct_prop.loc[:,tmp_order]
multiome_prop = multiome_prop.loc[:,tmp_order]

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 3))

ax = sns.heatmap(multiome_prop, vmin=0, vmax=0.7, cmap='coolwarm')
ax.grid(False)
ax.set_xlabel('Cluster', fontsize=16)
ax.set_ylabel('Time', fontsize=16)
ax.set_title('10X Multiome RNA', fontsize=20)

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 3))

ax = sns.heatmap(snmct_prop, vmin=0, vmax=0.7, cmap='coolwarm')
ax.grid(False)
ax.set_xlabel('Cluster', fontsize=16)
ax.set_ylabel('Time', fontsize=16)
ax.set_title('snmCT-seq RNA', fontsize=20)

In [ ]:
multiome_prop['modality'] = '10X Multiome'
snmct_prop['modality'] = 'snmCT-seq'
merged = pd.concat([multiome_prop, snmct_prop])
merged.head()

In [ ]:
tmp = merged[merged['modality'] == 'snmCT-seq']
tmp.sum(axis=1)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure1'
merged.to_csv(f'{outdir}/multiome_snmct_clust_time_prop.csv', sep='\t')

In [ ]:
multiome_prop['time_num'] = multiome_prop.index
mult_melt = multiome_prop.melt(id_vars='time_num', value_vars=multiome_prop.columns[:-2])

In [ ]:
snmct_prop['time_num'] = snmct_prop.index
snmct_melt = snmct_prop.melt(id_vars='time_num', value_vars=snmct_prop.columns[:-2])

In [ ]:
mult_melt.head()

In [ ]:
mult_melt['assay'] = 'multiome'
snmct_melt['assay'] = 'snmct'

In [ ]:
merged = pd.concat([mult_melt, snmct_melt])

In [ ]:
merged.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(4, 4))

i = 0

times = sorted(merged['time_num'].unique())
x_lab = 'variable'
y_lab = 'value'
hue = 'assay'

t = times[i]
tmp_plot = merged[merged['time_num'] == t]
ax = sns.barplot(data=tmp_plot, x=x_lab, y=y_lab, hue=hue)
ax.set_xlabel('Cluster', fontsize=16)
ax.set_ylabel('# cells', fontsize=16)
ax.set_title(t, fontsize=16)
ax.set_ylim(0, 1)
ax.tick_params(axis='x', labelsize=14, labelrotation=45, pad=10)
ax.grid(False)

sns.move_legend(ax, "upper left", bbox_to_anchor=(1.05, 1))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(24, 10))

times = sorted(merged['time_num'].unique())
x_lab = 'variable'
y_lab = 'value'
hue = 'assay'

for i, ax in enumerate(axes.flatten()):
    t = times[i]
    tmp_plot = merged[merged['time_num'] == t]
    sns.barplot(data=tmp_plot, x=x_lab, y=y_lab, hue=hue, ax=ax)
    ax.set_xlabel('Cluster', fontsize=16)
    ax.set_ylabel('# cells', fontsize=16)
    ax.set_title(f'Day {t}', fontsize=16)
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', labelsize=14, labelrotation=45, pad=10)
    ax.get_legend().remove()
    ax.grid(False)

plt.subplots_adjust(wspace=0.4, hspace=0.6)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure1'
merged.to_csv(f'{outdir}/xgboost_per_timepoint_clust_dist.csv', sep='\t')

### plot on mCG umap now

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/'
adata = sc.read_h5ad(f'{indir}/h5ad/chrom100k/merged_less_obs.h5ad')

In [ ]:
adata

In [ ]:
adata.obs['rna_cluster'] = snmct.reindex(adata.obs.index)['cluster']
final_adata = adata[~adata.obs['rna_cluster'].isna()]

In [ ]:
sc.pl.umap(final_adata, color=['rna_cluster'])

In [ ]:
def cluster_small_multiples(adata, clust_key, size=10, ncols=2,
                            frameon=False, legend_loc=None, **kwargs):
    tmp = adata.copy()

    for i,clust in enumerate(adata.obs[clust_key].cat.categories):
        tmp.obs[clust] = adata.obs[clust_key].isin([clust]).astype('category')
        tmp.uns[clust+'_colors'] = ['#d3d3d3', adata.uns[clust_key+'_colors'][i]]

    sc.pl.umap(tmp, groups=tmp.obs[clust].cat.categories[1:].values,
               color=adata.obs[clust_key].cat.categories.tolist(),
               size=size, frameon=frameon, legend_loc=legend_loc,
               ncols=ncols, **kwargs)

In [ ]:
cluster_small_multiples(final_adata, 'rna_cluster', ncols=4)

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/'
adatas = []
for donor in donors:
    tmp_adata = sc.read_h5ad(f'{indir}/h5ad/chrom100k/{donor}.h5ad')
    tmp_adata.obs['rna_cluster'] = snmct.reindex(tmp_adata.obs.index)['cluster']
    tmp_adata = tmp_adata[~tmp_adata.obs['rna_cluster'].isna()]
    tmp_adata.obs = tmp_adata.obs[['line', 'time', 'rna_cluster']]
    tmp_adata.obs['rna_cluster'].astype('category')
    adatas.append(tmp_adata)

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.pl.umap(tmp_adata, color=['rna_cluster'])

In [ ]:
for tmp_adata in adatas:
    sc.pl.umap(tmp_adata, color=['rna_cluster'], title='')

In [ ]:
sc.set_figure_params(figsize=(3,3), frameon=False)
for tmp_adata in adatas:
    cluster_small_multiples(tmp_adata, 'rna_cluster', ncols=4)

In [ ]:
!date